# Agente DJ 🎧

Você vai construir um agente que monta playlists a partir de uma frase sua.
Você escreve "quero treinar de manhã" e recebe 8 músicas de verdade, tiradas de uma base com mais de 5 mil faixas.
São poucas linhas de código: a inteligência artificial entra só para entender o seu pedido.

## Passo 1: a base

A base `musicas_diva.csv` já está pronta e limpa — a aula de limpeza de dados fica para outro dia.
Cada linha é uma música, com colunas como `titulo`, `artista`, `energia` e `popularidade`.

In [ ]:
import pandas as pd

musicas = pd.read_csv('data/musicas_diva.csv')

print(len(musicas))
musicas.head(3)

## Passo 2: o agente

Agora entra o Gemini, a inteligência artificial do Google.

O trabalho dele aqui é bem pequeno: ele lê a sua frase e devolve **um número só**, de 0 a 1,
dizendo qual é a energia da playlist que você pediu. "Quero treinar" vira algo perto de 0.9.
"Quero dormir" vira algo perto de 0.1.

Com esse número na mão, **o seu código** vai até a base e escolhe as músicas.

In [ ]:
import pandas as pd, os, logging
from dotenv import load_dotenv
from google import genai
logging.getLogger('google_genai').setLevel('ERROR')

musicas = pd.read_csv('data/musicas_diva.csv')
load_dotenv()
cliente = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def playlist(frase):
    resposta = cliente.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents='De 0 a 1, qual a energia da playlist para este pedido? '
                 'Responda só o número. Pedido: ' + frase)
    energia = float(resposta.text.strip())
    print('A IA entendeu: energia', energia)
    # pega as 150 músicas de energia mais parecida, e dessas fica com as 8 mais famosas
    perto = musicas.assign(d=(musicas['energia'] - energia).abs()).sort_values('d').head(150)
    return perto.sort_values('popularidade', ascending=False).head(8)[['titulo','artista','energia']]

playlist('quero treinar de manhã')

## Passo 3: a sua vez

Troque a frase abaixo pelo seu pedido, com as suas palavras.
Pode ser um clima, um momento do dia, um sentimento — o que você quiser ouvir agora.

In [ ]:
playlist('tô de coração partido')

## A regra de ouro

A inteligência artificial **não** escolhe nenhuma música. Olhe as duas linhas:

- `resposta = cliente.models.generate_content(...)` — aqui a IA entra, lê a sua frase e devolve um número.
- `perto = musicas.assign(...)` — aqui o **seu código** procura na base e escolhe as músicas.

É por isso que a playlist é sempre de músicas reais: elas saem da sua base, não da cabeça da IA.

Pronto, o seu agente está no ar. Aproveite a playlist 🎉

## Se der errado

| O que aparece na tela | O que fazer |
| --- | --- |
| `KeyError: 'GEMINI_API_KEY'` | O arquivo `.env` não foi encontrado. Ele precisa estar na pasta principal do projeto, com a linha `GEMINI_API_KEY=sua_chave`. |
| `ModuleNotFoundError` | Falta instalar as bibliotecas. No terminal: `pip install pandas google-genai python-dotenv` |
| `FileNotFoundError` | Este caderno precisa estar na mesma pasta que a pasta `data/`. |
| Mensagem com `quota` ou `429` | A chave passou do limite de uso do dia. Use o plano B abaixo e siga a aula normalmente. |
### Plano B: sem IA

Se a sua chave não estiver funcionando, rode a célula abaixo e escolha a energia na mão.
É exatamente a mesma busca de músicas — só o número vem de você, e não do Gemini.

In [ ]:
energia = 0.8   # troque aqui: 0.9 agitada, 0.5 média, 0.2 calma
musicas.assign(d=(musicas['energia']-energia).abs()).sort_values('d').head(150) \
       .sort_values('popularidade', ascending=False).head(8)[['titulo','artista','energia']]